# 06 · P3 闭环 — Postmortem（成交/丢单 → 偏差 → 知识回流）+ 客户记忆召回（援）

Quote → Won/Lost → 实际成本/交期 → 偏差分析 → knowledge_updates → 下次报价；客户历史作为 Fact memory 召回, 注入 Context 作证据与风险信号。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.postmortem import record_outcome, recall_customer_memory
ctrl = build_controller()
r = ctrl.run(email_text='50 pcs 6061 aluminum, 100x50x10mm, anodizing, IT7.',
             customer={'name':'ClosedLoop Demo Co','country':'US'}, destination_country='US')
cid = r['context_id']; final = r['quote'].get('final_price')
print('quote final=', final, 'state=', r['state'], 'customer_memory=', r['customer_memory'])

In [ ]:
pm = record_outcome(ctrl.crm, cid, 'won', actual_cost=round(final*1.15,2),
                    actual_leadtime_days=(r['commercial']['total_lead_time_days']+5), note='实际成本偏高')
print('deviation=', pm['deviation'])
print('knowledge_updates=', [k['type'] for k in pm['knowledge_updates']])

In [ ]:
m = recall_customer_memory(ctrl.crm, {'name':'ClosedLoop Demo Co','country':'US'})
print('is_new=', m['is_new'], 'signals=', [s['type'] for s in m['signals']])
print('history quotes=', m['history']['n'])
ctrl.crm.close()